# Dataset 2: demographics, hospital stay, and treatment EDA

This split contains age bands, weight, diagnosis, hospital stay, procedures, disposition, and insulin-related variables.

The notebook analyses this split independently. The original CSV is not changed; the cleaned copy is saved in `EDA_Dataset` for later joining on `encounter_id`.

## Setup

In [1]:
from pathlib import Path  # Import required library

import matplotlib.pyplot as plt  # Import required library
import pandas as pd  # Import required library
import seaborn as sns  # Import required library

sns.set_theme(style="whitegrid", context="notebook")  # Set the chart theme
pd.set_option("display.max_rows", 100)  # Configure dataframe display
pd.set_option("display.max_columns", 50)  # Configure dataframe display

In [2]:
def show_category_counts(series, title, top_n=15):  # Define the categorical-frequency chart helper
    counts = series.fillna("Missing").value_counts().head(top_n).sort_values()  # Calculate category frequencies
    fig, ax = plt.subplots(figsize=(10, max(3.5, 0.38 * len(counts))))  # Create the chart area
    ax.barh(counts.index.astype(str), counts.values, color="#4C78A8")  # Draw the horizontal bar chart
    ax.set_title(f"{title}: top {top_n} categories")  # Label the chart axis or title
    ax.set_xlabel("Encounter count")  # Label the chart axis or title
    ax.set_ylabel("")  # Label the chart axis or title
    for y, value in enumerate(counts.values):  # Add value labels to each bar
        ax.text(value, y, f" {value:,}", va="center", fontsize=8)  # Run this EDA or data-cleaning step
    plt.tight_layout()  # Adjust chart spacing
    plt.show()  # Display the chart


def show_numeric_distribution(series, title, discrete=False):  # Define the numeric-distribution chart helper
    values = series.dropna()  # Remove missing numeric values before plotting
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))  # Create the chart area
    sns.histplot(values, discrete=discrete, ax=axes[0], color="#4C78A8")  # Plot the feature distribution
    axes[0].set_title(f"{title}: distribution")  # Configure the numeric chart
    axes[0].set_xlabel(title)  # Configure the numeric chart
    sns.boxplot(x=values, ax=axes[1], color="#72B7B2")  # Plot the feature box plot
    axes[1].set_title(f"{title}: box plot")  # Configure the numeric chart
    axes[1].set_xlabel(title)  # Configure the numeric chart
    plt.tight_layout()  # Adjust chart spacing
    plt.show()  # Display the chart

## Load the source data

In [3]:
source_path = Path("divided_datasets") / "dataset_2.csv"  # Set the path to the source CSV
df_raw = pd.read_csv(source_path, na_values=["?"], keep_default_na=True, low_memory=False)  # Load the source CSV and convert question marks to missing values

## Initial inspection

In [4]:
df_raw.head()  # Inspect the original dataframe

,encounter_id,diag_3,num_procedures,weight,acarbose,time_in_hospital,age,discharge_disposition_id,insulin
0,2278392,NaN,0,NaN,No,1,[0-10),25,No
1,149190,255,0,NaN,No,3,[10-20),1,Up
2,64410,V27,5,NaN,No,2,[20-30),1,No
3,500364,403,1,NaN,No,2,[30-40),1,Up
4,16680,250,0,NaN,No,1,[40-50),1,Steady


In [5]:
df_raw.info()  # Inspect the original dataframe

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 9 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   encounter_id              101766 non-null  int64
 1   diag_3                    100343 non-null  str  
 2   num_procedures            101766 non-null  int64
 3   weight                    3197 non-null    str  
 4   acarbose                  101766 non-null  str  
 5   time_in_hospital          101766 non-null  int64
 6   age                       101766 non-null  str  
 7   discharge_disposition_id  101766 non-null  int64
 8   insulin                   101766 non-null  str  
dtypes: int64(4), str(5)
memory usage: 8.5 MB


In [6]:
df_raw.shape  # Inspect the original dataframe

(101766, 9)

In [7]:
df_raw.describe()  # Inspect the original dataframe

,encounter_id,num_procedures,time_in_hospital,discharge_disposition_id
count,1.017660e+05,101766.000000,101766.000000,101766.000000
mean,1.652016e+08,1.339730,4.395987,3.715642
std,1.026403e+08,1.705807,2.985108,5.280166
min,1.252200e+04,0.000000,1.000000,1.000000
25%,8.496119e+07,0.000000,2.000000,1.000000
50%,1.523890e+08,1.000000,4.000000,1.000000
75%,2.302709e+08,2.000000,6.000000,4.000000
max,4.438672e+08,6.000000,14.000000,28.000000


In [8]:
df_raw.isna().sum().sort_values(ascending=False).to_frame(name="missing_count")  # Inspect the original dataframe

,missing_count
weight,98569
diag_3,1423
encounter_id,0
num_procedures,0
acarbose,0
time_in_hospital,0
age,0
discharge_disposition_id,0
insulin,0


In [9]:
df_raw.duplicated().sum()  # Inspect the original dataframe

np.int64(0)

In [10]:
df_raw["encounter_id"].duplicated().sum()  # Run this EDA or data-cleaning step

np.int64(0)

In [11]:
df_raw["encounter_id"].is_unique  # Run this EDA or data-cleaning step

True

In [12]:
df_raw.nunique(dropna=False).sort_values().to_frame(name="unique_values_including_missing")  # Inspect the original dataframe

,unique_values_including_missing
acarbose,4
insulin,4
num_procedures,7
age,10
weight,10
time_in_hospital,14
discharge_disposition_id,26
diag_3,790
encounter_id,101766


## Source-field checks before cleaning

### Weight before removal

In [13]:
df_raw["weight"].isna().mean()  # Run this EDA or data-cleaning step

np.float64(0.9685847925633315)

In [14]:
df_raw["weight"].value_counts(dropna=False).head(12).to_frame(name="count")  # Run this EDA or data-cleaning step

,count
weight,
NaN,98569
[75-100),1336
[50-75),897
[100-125),625
[125-150),145
[25-50),97
[0-25),48
[150-175),35
[175-200),11


In [15]:
show_category_counts(df_raw["weight"], "Weight before removal", top_n=12)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_7368\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Age interval before expansion

In [16]:
df_raw["age"].value_counts(dropna=False).head(12).to_frame(name="count")  # Run this EDA or data-cleaning step

,count
age,
[70-80),26068
[60-70),22483
[50-60),17256
[80-90),17197
[40-50),9685
[30-40),3775
[90-100),2793
[20-30),1657
[10-20),691


In [17]:
show_category_counts(df_raw["age"], "Age interval before expansion", top_n=12)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_7368\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


## Cleaning decisions

- Weight removed: weight is removed because 96.86% of its values are missing in the source.
- Age interval expanded: The source age interval is replaced by age_min and age_max, using the lower and upper bounds of the original interval.
- Third diagnosis: A missingness flag is retained and missing diagnosis codes are labeled Unknown for visualization.
- Rows and join key: No rows are removed. encounter_id remains unchanged and unique for the later join.

In [18]:
df = df_raw.copy()  # Copy the source dataframe before cleaning

text_columns = df.select_dtypes(include=["object", "string"]).columns  # Identify text columns for normalization
for column in text_columns:  # Clean each text column
    df[column] = df[column].astype("string").str.strip().replace("", pd.NA)  # Strip whitespace and convert blank text to missing values

weight_missing_rate = df["weight"].isna().mean()  # Calculate the weight missing-value rate
assert weight_missing_rate > 0.95, "weight should only be removed when overwhelmingly missing"  # Validate the cleaning result

df["diag_3_missing"] = df["diag_3"].isna()  # Create or handle the diagnosis missingness information
df["diag_3"] = df["diag_3"].fillna("Unknown")  # Create or handle the diagnosis missingness information

age_bounds = df["age"].str.extract(r"^\[(\d+)-(\d+)\)$").astype("Int64")  # Extract lower and upper values from age intervals
df["age_min"] = age_bounds[0]  # Create the age interval lower-bound column
df["age_max"] = age_bounds[1]  # Create the age interval upper-bound column
assert df[["age_min", "age_max"]].notna().all().all(), "Unparsed age interval found"  # Validate the cleaning result
assert (df["age_min"] < df["age_max"]).all(), "Invalid age interval found"  # Validate the cleaning result

columns_removed = ["weight", "age"]  # Record columns removed during cleaning
df = df.drop(columns=columns_removed)  # Remove the documented uninformative columns

assert len(df) == len(df_raw), "No encounter rows should be removed"  # Validate the cleaning result
assert df["encounter_id"].notna().all(), "The join key must not be missing"  # Validate the cleaning result
assert df["encounter_id"].is_unique, "The join key must remain unique"  # Validate the cleaning result
assert df.columns.is_unique, "Duplicate column names are not allowed"  # Validate the cleaning result

In [19]:
cleaning_summary = pd.DataFrame({  # Create the cleaning-decision summary
    "decision": ["Weight removed","Age interval expanded","Third diagnosis","Rows and join key","Columns removed"],  # Add a cleaning-summary field
    "detail": ["weight is removed because 96.86% of its values are missing in the source.","The source age interval is replaced by age_min and age_max, using the lower and upper bounds of the original interval.","A missingness flag is retained and missing diagnosis codes are labeled Unknown for visualization.","No rows are removed. encounter_id remains unchanged and unique for the later join.","Columns removed after review: weight, age"]  # Add a cleaning-summary field
})  # Run this EDA or data-cleaning step
cleaning_summary  # Create the cleaning-decision summary

,decision,detail
0,Weight removed,weight is removed because 96.86% of its values...
1,Age interval expanded,The source age interval is replaced by age_min...
2,Third diagnosis,A missingness flag is retained and missing dia...
3,Rows and join key,No rows are removed. encounter_id remains unch...
4,Columns removed,"Columns removed after review: weight, age"


## Post-cleaning validation

In [20]:
df.head()  # Run this EDA or data-cleaning step

,encounter_id,diag_3,num_procedures,acarbose,time_in_hospital,discharge_disposition_id,insulin,diag_3_missing,age_min,age_max
0,2278392,Unknown,0,No,1,25,No,True,0,10
1,149190,255,0,No,3,1,Up,False,10,20
2,64410,V27,5,No,2,1,No,False,20,30
3,500364,403,1,No,2,1,Up,False,30,40
4,16680,250,0,No,1,1,Steady,False,40,50


In [21]:
df.info()  # Run this EDA or data-cleaning step

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 10 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   diag_3                    101766 non-null  string
 2   num_procedures            101766 non-null  int64 
 3   acarbose                  101766 non-null  string
 4   time_in_hospital          101766 non-null  int64 
 5   discharge_disposition_id  101766 non-null  int64 
 6   insulin                   101766 non-null  string
 7   diag_3_missing            101766 non-null  bool  
 8   age_min                   101766 non-null  Int64 
 9   age_max                   101766 non-null  Int64 
dtypes: Int64(2), bool(1), int64(4), string(3)
memory usage: 8.1 MB


In [22]:
df.shape  # Run this EDA or data-cleaning step

(101766, 10)

In [23]:
df.describe()  # Run this EDA or data-cleaning step

,encounter_id,num_procedures,time_in_hospital,discharge_disposition_id,age_min,age_max
count,1.017660e+05,101766.000000,101766.000000,101766.000000,101766.0,101766.0
mean,1.652016e+08,1.339730,4.395987,3.715642,60.967022,70.967022
std,1.026403e+08,1.705807,2.985108,5.280166,15.940838,15.940838
min,1.252200e+04,0.000000,1.000000,1.000000,0.0,10.0
25%,8.496119e+07,0.000000,2.000000,1.000000,50.0,60.0
50%,1.523890e+08,1.000000,4.000000,1.000000,60.0,70.0
75%,2.302709e+08,2.000000,6.000000,4.000000,70.0,80.0
max,4.438672e+08,6.000000,14.000000,28.000000,90.0,100.0


In [24]:
df.isna().sum().sort_values(ascending=False).to_frame(name="missing_count_after_cleaning")  # Run this EDA or data-cleaning step

,missing_count_after_cleaning
encounter_id,0
diag_3,0
num_procedures,0
acarbose,0
time_in_hospital,0
discharge_disposition_id,0
insulin,0
diag_3_missing,0
age_min,0
age_max,0


In [25]:
df["encounter_id"].nunique()  # Inspect the selected cleaned column

101766

## Univariate EDA after cleaning

### Number of procedures

In [26]:
df["num_procedures"].describe()  # Inspect the selected cleaned column

count    101766.000000
mean          1.339730
std           1.705807
min           0.000000
25%           0.000000
50%           1.000000
75%           2.000000
max           6.000000
Name: num_procedures, dtype: float64

In [27]:
show_numeric_distribution(df["num_procedures"], "Number of procedures", discrete=True)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_7368\4155684497.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Time in hospital (days)

In [28]:
df["time_in_hospital"].describe()  # Inspect the selected cleaned column

count    101766.000000
mean          4.395987
std           2.985108
min           1.000000
25%           2.000000
50%           4.000000
75%           6.000000
max          14.000000
Name: time_in_hospital, dtype: float64

In [29]:
show_numeric_distribution(df["time_in_hospital"], "Time in hospital (days)", discrete=True)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_7368\4155684497.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Age interval lower bound

In [30]:
df["age_min"].describe()  # Create the age interval lower-bound column

count     101766.0
mean     60.967022
std      15.940838
min            0.0
25%           50.0
50%           60.0
75%           70.0
max           90.0
Name: age_min, dtype: Float64

In [31]:
show_numeric_distribution(df["age_min"], "Age interval lower bound", discrete=True)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_7368\4155684497.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Age interval upper bound

In [32]:
df["age_max"].describe()  # Create the age interval upper-bound column

count     101766.0
mean     70.967022
std      15.940838
min           10.0
25%           60.0
50%           70.0
75%           80.0
max          100.0
Name: age_max, dtype: Float64

In [33]:
show_numeric_distribution(df["age_max"], "Age interval upper bound", discrete=True)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_7368\4155684497.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Third diagnosis code

In [34]:
df["diag_3"].value_counts(dropna=False).head(20).to_frame(name="count")  # Create or handle the diagnosis missingness information

,count
diag_3,
250,11555
401,8289
276,5175
428,4577
427,3955
414,3664
496,2605
403,2357
585,1992


In [35]:
show_category_counts(df["diag_3"], "Third diagnosis code", top_n=20)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_7368\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Acarbose status

In [36]:
df["acarbose"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
acarbose,
No,101458
Steady,295
Up,10
Down,3


In [37]:
show_category_counts(df["acarbose"], "Acarbose status", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_7368\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Discharge disposition ID

In [38]:
df["discharge_disposition_id"].value_counts(dropna=False).head(30).to_frame(name="count")  # Inspect the selected cleaned column

,count
discharge_disposition_id,
1,60234
3,13954
6,12902
18,3691
2,2128
22,1993
11,1642
5,1184
25,989


In [39]:
show_category_counts(df["discharge_disposition_id"], "Discharge disposition ID", top_n=30)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_7368\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Insulin status

In [40]:
df["insulin"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
insulin,
No,47383
Steady,30849
Down,12218
Up,11316


In [41]:
show_category_counts(df["insulin"], "Insulin status", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_7368\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


## Numeric relationships

In [42]:
numeric_columns = ["num_procedures", "time_in_hospital", "age_min", "age_max"]  # Select numeric columns for relationship analysis
df[numeric_columns].corr()  # Inspect the selected cleaned column

,num_procedures,time_in_hospital,age_min,age_max
num_procedures,1.000000,0.191472,-0.030104,-0.030104
time_in_hospital,0.191472,1.000000,0.107515,0.107515
age_min,-0.030104,0.107515,1.000000,1.000000
age_max,-0.030104,0.107515,1.000000,1.000000


In [43]:
correlation = df[numeric_columns].corr()  # Calculate the numeric correlation matrix
plt.figure(figsize=(7, 5))  # Create the chart area
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="Blues", vmin=-1, vmax=1)  # Plot the numeric correlation heatmap
plt.title("Correlation between numeric features")  # Set the chart title
plt.tight_layout()  # Adjust chart spacing
plt.show()  # Display the chart

C:\Users\anind\AppData\Local\Temp\ipykernel_7368\1070095895.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


## Save the cleaned split

In [44]:
output_dir = Path("EDA_Dataset")  # Set the cleaned-data output folder
output_dir.mkdir(exist_ok=True)  # Set the cleaned-data output folder
output_path = output_dir / "dataset_2.csv"  # Set the cleaned CSV output path
df.to_csv(output_path, index=False)  # Save the cleaned dataframe
print(f"Saved {df.shape[0]:,} rows and {df.shape[1]:,} columns to {output_path}")  # Report the saved dataset details

Saved 101,766 rows and 10 columns to EDA_Dataset\dataset_2.csv


In [45]:
saved_df = pd.read_csv(output_path)  # Reload the saved CSV for verification
assert saved_df.shape == df.shape, "Saved CSV shape does not match the cleaned dataframe"  # Validate the cleaning result
saved_df.head()  # Reload the saved CSV for verification

,encounter_id,diag_3,num_procedures,acarbose,time_in_hospital,discharge_disposition_id,insulin,diag_3_missing,age_min,age_max
0,2278392,Unknown,0,No,1,25,No,True,0,10
1,149190,255,0,No,3,1,Up,False,10,20
2,64410,V27,5,No,2,1,No,False,20,30
3,500364,403,1,No,2,1,Up,False,30,40
4,16680,250,0,No,1,1,Steady,False,40,50
